In [1]:
## 셀 1: 데이터 불러오기

import geopandas as gpd
import pandas as pd

shadow_11 = gpd.read_file('bundang_shadow_20240805_1100_4326.gpkg')
shadow_17 = gpd.read_file('bundang_shadow_20240805_1700_4326.gpkg')

difficult_10 = pd.read_csv('difficult_segments_final10.csv')
print(f"그림자(11시): {len(shadow_11)}개, 그림자(17시): {len(shadow_17)}개")
print(f"곤란구간: {len(difficult_10)}개")

그림자(11시): 21793개, 그림자(17시): 21793개
곤란구간: 10개


In [4]:
## 셀 2 재수정: 곤란구간 buffer와 그림자 polygon 겹침 비율 계산

road_gdf = gpd.read_file('03._상세도로망_lev6.geojson')
osm_roads = gpd.read_file('gis_osm_roads_free_1.shp', bbox=(127.05, 37.33, 127.16, 37.43))
footway_types = ['footway', 'path', 'pedestrian', 'steps']
sidewalk = osm_roads[osm_roads['fclass'].isin(footway_types)].copy()

road_gdf_calc = road_gdf[['link_id', 'geometry']].copy()
sidewalk_calc = sidewalk[['osm_id', 'geometry']].copy()
sidewalk_calc['link_id'] = 'osm_' + sidewalk_calc['osm_id'].astype(str)
sidewalk_calc = sidewalk_calc[['link_id', 'geometry']]
all_links_gdf = pd.concat([road_gdf_calc, sidewalk_calc], ignore_index=True)

# difficult_10에서 geometry 제외하고 필요한 컬럼만 (link_id, link_hsi만)
difficult_10_clean = difficult_10[['link_id', 'link_hsi']].copy()
difficult_10_clean['link_id'] = difficult_10_clean['link_id'].astype(str)

all_links_gdf['link_id'] = all_links_gdf['link_id'].astype(str)

difficult_geom = all_links_gdf[all_links_gdf['link_id'].isin(difficult_10_clean['link_id'])].copy()
difficult_geom = difficult_geom.merge(difficult_10_clean, on='link_id', how='left')

print("컬럼 확인:", difficult_geom.columns.tolist())
print("타입 확인:", type(difficult_geom))

# 5186으로 변환 후 12m 버퍼
difficult_geom_5186 = difficult_geom.to_crs(epsg=5186)
difficult_geom_5186['buffer_geom'] = difficult_geom_5186.geometry.buffer(12)
difficult_buffer = difficult_geom_5186.set_geometry('buffer_geom')[['link_id', 'link_hsi', 'buffer_geom']]
difficult_buffer['buffer_area'] = difficult_buffer.geometry.area

print(difficult_buffer[['link_id','link_hsi','buffer_area']])

컬럼 확인: ['link_id', 'geometry', 'link_hsi']
타입 확인: <class 'geopandas.geodataframe.GeoDataFrame'>
     link_id  link_hsi  buffer_area
0  571500274  0.902470  4864.398783
1  571500307  0.899820  2302.142443
2  571500575  0.865877  2352.514934
3  571541307  0.887483  1912.455200
4  571541880  0.916116  1526.012814
5  571541906  0.817914  1236.238988
6  571541909  0.904698  3633.992056
7  571542735  0.821341  2792.692379
8  571549132  0.809193  1065.316390
9  571551868  0.822205  1911.607916


In [5]:
## 셀 3: 그림자 커버리지 비율 계산

shadow_11_5186 = shadow_11.to_crs(epsg=5186)
shadow_17_5186 = shadow_17.to_crs(epsg=5186)

def calc_shadow_coverage(buffer_gdf, shadow_gdf, label):
    overlay = gpd.overlay(buffer_gdf, shadow_gdf[['geometry']], how='intersection')
    overlay['shadow_area'] = overlay.geometry.area
    coverage = overlay.groupby('link_id')['shadow_area'].sum().reset_index()
    coverage.columns = ['link_id', f'shadow_area_{label}']
    return coverage

cov_11 = calc_shadow_coverage(difficult_buffer, shadow_11_5186, '11시')
cov_17 = calc_shadow_coverage(difficult_buffer, shadow_17_5186, '17시')

result = difficult_buffer[['link_id','link_hsi','buffer_area']].merge(cov_11, on='link_id', how='left').merge(cov_17, on='link_id', how='left')
result['shadow_area_11시'] = result['shadow_area_11시'].fillna(0)
result['shadow_area_17시'] = result['shadow_area_17시'].fillna(0)
result['그늘비율_11시'] = result['shadow_area_11시'] / result['buffer_area']
result['그늘비율_17시'] = result['shadow_area_17시'] / result['buffer_area']

print(result[['link_id','link_hsi','그늘비율_11시','그늘비율_17시']].sort_values('link_hsi', ascending=False))

     link_id  link_hsi  그늘비율_11시  그늘비율_17시
4  571541880  0.916116  0.177430  1.094724
6  571541909  0.904698  0.522521  1.195281
0  571500274  0.902470  0.336762  1.839828
1  571500307  0.899820  0.482870  0.693339
3  571541307  0.887483  0.000000  0.110672
2  571500575  0.865877  0.297563  0.231623
9  571551868  0.822205  0.000000  0.000000
7  571542735  0.821341  0.432805  0.239124
5  571541906  0.817914  0.267513  1.124210
8  571549132  0.809193  0.104501  0.000000


In [8]:
## 수정: unary_union 속성 방식으로 변경

shadow_11_dissolved = shadow_11_5186.geometry.unary_union
shadow_17_dissolved = shadow_17_5186.geometry.unary_union

shadow_11_single = gpd.GeoDataFrame(geometry=[shadow_11_dissolved], crs='EPSG:5186')
shadow_17_single = gpd.GeoDataFrame(geometry=[shadow_17_dissolved], crs='EPSG:5186')

print("dissolve 완료")

dissolve 완료


In [9]:
## 그림자 커버리지 비율 계산 (dissolve된 그림자 기준)

cov_11 = calc_shadow_coverage(difficult_buffer, shadow_11_single, '11시')
cov_17 = calc_shadow_coverage(difficult_buffer, shadow_17_single, '17시')

result = difficult_buffer[['link_id','link_hsi','buffer_area']].merge(cov_11, on='link_id', how='left').merge(cov_17, on='link_id', how='left')
result['shadow_area_11시'] = result['shadow_area_11시'].fillna(0)
result['shadow_area_17시'] = result['shadow_area_17시'].fillna(0)
result['그늘비율_11시'] = (result['shadow_area_11시'] / result['buffer_area']).clip(upper=1.0)
result['그늘비율_17시'] = (result['shadow_area_17시'] / result['buffer_area']).clip(upper=1.0)

print(result[['link_id','link_hsi','그늘비율_11시','그늘비율_17시']].sort_values('link_hsi', ascending=False))

     link_id  link_hsi  그늘비율_11시  그늘비율_17시
4  571541880  0.916116  0.177430  0.897359
6  571541909  0.904698  0.478133  0.934023
0  571500274  0.902470  0.316443  0.999993
1  571500307  0.899820  0.482870  0.557719
3  571541307  0.887483  0.000000  0.110672
2  571500575  0.865877  0.297563  0.181821
9  571551868  0.822205  0.000000  0.000000
7  571542735  0.821341  0.357434  0.204146
5  571541906  0.817914  0.160036  1.000000
8  571549132  0.809193  0.104501  0.000000


In [11]:
## grid_HSI_final_merged.csv에 필요한 컬럼 있는지 확인

df_check = pd.read_csv('grid_HSI_final_merged.csv', nrows=3)
target = ['wd_v_pop','we_v_pop','wd_w_pop','we_w_pop','w_pop_sd','v_pop_sd']
for c in target:
    print(c, ':', c in df_check.columns)

wd_v_pop : True
we_v_pop : True
wd_w_pop : True
we_w_pop : True
w_pop_sd : False
v_pop_sd : False


In [12]:
## 확인: 09번 원본 파일에 표준편차 컬럼 있는지

df_wd = pd.read_csv('09._주중_서비스인구.csv', nrows=3)
df_we = pd.read_csv('09._주말_서비스인구.csv', nrows=3)

print('주중 서비스인구 컬럼:', df_wd.columns.tolist())
print()
print('주말 서비스인구 컬럼:', df_we.columns.tolist())

주중 서비스인구 컬럼: ['gbn', 'STD_YM', 'hw', 'w_pop', 'v_pop', 'lon', 'lat']

주말 서비스인구 컬럼: ['gbn', 'STD_YM', 'hw', 'w_pop', 'v_pop', 'lon', 'lat']


In [13]:
## 표준편차(주중-주말 편차) 직접 계산

import numpy as np

# gbn이 grid_id인지 확인 필요
df_wd = pd.read_csv('09._주중_서비스인구.csv')
df_we = pd.read_csv('09._주말_서비스인구.csv')

print(df_wd['gbn'].head())
print(df_wd['gbn'].dtype)

0    경기도 성남시
1    경기도 성남시
2    경기도 성남시
3    경기도 성남시
4    경기도 성남시
Name: gbn, dtype: object
object


In [14]:
## hw 컬럼 확인 (격자ID 후보)

print(df_wd['hw'].head(10))
print(df_wd['hw'].dtype)
print(df_wd['hw'].nunique())

0    H
1    H
2    H
3    H
4    H
5    H
6    H
7    H
8    H
9    H
Name: hw, dtype: object
object
1


In [15]:
## 좌표 기반 격자 매칭

import geopandas as gpd
from shapely.geometry import Point

df_wd = pd.read_csv('09._주중_서비스인구.csv')
df_we = pd.read_csv('09._주말_서비스인구.csv')

print("주중 데이터 행 수:", len(df_wd))
print("주말 데이터 행 수:", len(df_we))
print(df_wd[['lon','lat','w_pop','v_pop']].describe())

# 좌표를 geometry로 변환
df_wd['geometry'] = df_wd.apply(lambda r: Point(r['lon'], r['lat']), axis=1)
gdf_wd = gpd.GeoDataFrame(df_wd, geometry='geometry', crs='EPSG:4326')

df_we['geometry'] = df_we.apply(lambda r: Point(r['lon'], r['lat']), axis=1)
gdf_we = gpd.GeoDataFrame(df_we, geometry='geometry', crs='EPSG:4326')

# 04번 격자 불러오기 (폴리곤)
grid_04 = gpd.read_file('04._격자_30mx30m.geojson')
print("\n04번 격자 컬럼:", grid_04.columns.tolist())

# 공간조인
gdf_wd_matched = gpd.sjoin(gdf_wd, grid_04[['grid_id','geometry']], how='left', predicate='within')
gdf_we_matched = gpd.sjoin(gdf_we, grid_04[['grid_id','geometry']], how='left', predicate='within')

print("\n주중 매칭 성공:", gdf_wd_matched['grid_id'].notna().sum(), "/", len(gdf_wd_matched))
print("주말 매칭 성공:", gdf_we_matched['grid_id'].notna().sum(), "/", len(gdf_we_matched))

주중 데이터 행 수: 465283
주말 데이터 행 수: 487019
                 lon            lat          w_pop          v_pop
count  465283.000000  465283.000000  465283.000000  465283.000000
mean      127.121510      37.407981       0.039204       0.174084
std         0.033407       0.034385       0.116273       0.497873
min       127.028275      37.333552       0.000000       0.000000
25%       127.100280      37.380479       0.000324       0.009506
50%       127.123133      37.408186       0.005750       0.031328
75%       127.146422      37.436284       0.030573       0.142799
max       127.195202      37.474352       3.022044      14.585000

04번 격자 컬럼: ['grid_id', 'geometry']

주중 매칭 성공: 133462 / 465283
주말 매칭 성공: 134946 / 487019


In [16]:
## STD_YM 값 확인 - 여러 시점이 쌓여있는지

print(df_wd['STD_YM'].value_counts())
print()
print(df_wd['STD_YM'].nunique())

STD_YM
202411    39971
202412    39322
202410    39301
202407    39131
202408    38985
202409    38944
202406    38895
202405    38596
202404    38475
202403    38127
202402    37832
202401    37704
Name: count, dtype: int64

12


In [17]:
## 5~9월 필터링 + 격자 매칭 + 표준편차 계산

# 5~9월 필터링 (202405~202409)
months_target = [202405, 202406, 202407, 202408, 202409]

df_wd_summer = df_wd[df_wd['STD_YM'].isin(months_target)].copy()
df_we_summer = df_we[df_we['STD_YM'].isin(months_target)].copy()

print(f"주중 5~9월 필터링 후: {len(df_wd_summer)}행")
print(f"주말 5~9월 필터링 후: {len(df_we_summer)}행")

# 좌표 -> geometry 변환
df_wd_summer['geometry'] = df_wd_summer.apply(lambda r: Point(r['lon'], r['lat']), axis=1)
gdf_wd_summer = gpd.GeoDataFrame(df_wd_summer, geometry='geometry', crs='EPSG:4326')

df_we_summer['geometry'] = df_we_summer.apply(lambda r: Point(r['lon'], r['lat']), axis=1)
gdf_we_summer = gpd.GeoDataFrame(df_we_summer, geometry='geometry', crs='EPSG:4326')

# 격자 매칭
gdf_wd_matched = gpd.sjoin(gdf_wd_summer, grid_04[['grid_id','geometry']], how='inner', predicate='within')
gdf_we_matched = gpd.sjoin(gdf_we_summer, grid_04[['grid_id','geometry']], how='inner', predicate='within')

print(f"\n주중 매칭 성공: {len(gdf_wd_matched)}행")
print(f"주말 매칭 성공: {len(gdf_we_matched)}행")

# grid_id별 w_pop, v_pop 평균 (같은 격자 내 여러 월 데이터 -> 하나의 대표값)
wd_grid = gdf_wd_matched.groupby('grid_id').agg(wd_w_pop_new=('w_pop','mean'), wd_v_pop_new=('v_pop','mean')).reset_index()
we_grid = gdf_we_matched.groupby('grid_id').agg(we_w_pop_new=('w_pop','mean'), we_v_pop_new=('v_pop','mean')).reset_index()

# 주중-주말 merge
pop_merged = wd_grid.merge(we_grid, on='grid_id', how='outer')

# 표준편차 계산 (주중값, 주말값 두 개의 std)
pop_merged['w_pop_sd'] = pop_merged[['wd_w_pop_new','we_w_pop_new']].std(axis=1)
pop_merged['v_pop_sd'] = pop_merged[['wd_v_pop_new','we_v_pop_new']].std(axis=1)

print(f"\n최종 격자 수: {len(pop_merged)}")
print(pop_merged.describe())

pop_merged.to_csv('weekday_weekend_pop_sd.csv', index=False, encoding='utf-8-sig')
print("\n저장 완료: weekday_weekend_pop_sd.csv")

주중 5~9월 필터링 후: 194551행
주말 5~9월 필터링 후: 203855행

주중 매칭 성공: 55669행
주말 매칭 성공: 56296행

최종 격자 수: 11331
            grid_id  wd_w_pop_new  wd_v_pop_new  we_w_pop_new  we_v_pop_new  \
count  11331.000000  11184.000000  11184.000000  11300.000000  11300.000000   
mean   38075.331215      0.070426      0.299339      0.242258      0.314639   
std     9280.687043      0.176586      0.719613      0.710499      0.799188   
min    21907.000000      0.000000      0.000000      0.000000      0.000000   
25%    29965.000000      0.006686      0.034793      0.006330      0.018110   
50%    38220.000000      0.021454      0.101871      0.025621      0.063556   
75%    46098.000000      0.050616      0.228323      0.104563      0.230121   
max    54024.000000      2.428410     10.419546     10.402057     16.479942   

           w_pop_sd      v_pop_sd  
count  11153.000000  11153.000000  
mean       0.131149      0.096412  
std        0.422431      0.190473  
min        0.000000      0.000000  
25%        

In [18]:
## 4개 모델 스코어링

pop_merged_full = pop_merged.dropna(subset=['wd_v_pop_new','we_v_pop_new','wd_w_pop_new','we_w_pop_new','v_pop_sd','w_pop_sd'])

# 1. 분모 0 필터링 (df_safe)
df_safe = pop_merged_full[
    (pop_merged_full['wd_v_pop_new'] > 0) &
    (pop_merged_full['we_v_pop_new'] + pop_merged_full['we_w_pop_new'] > 0) &
    (pop_merged_full['w_pop_sd'] > 0)
].copy()

print(f"df_safe 격자 수: {len(df_safe)} / {len(pop_merged_full)}")

# 2. 파생 피처 계산
df_safe['v_pop_diff'] = df_safe['we_v_pop_new'] - df_safe['wd_v_pop_new']
df_safe['v_pop_ratio'] = df_safe['we_v_pop_new'] / df_safe['wd_v_pop_new']
df_safe['weekend_pure_v_ratio'] = df_safe['we_v_pop_new'] / (df_safe['we_v_pop_new'] + df_safe['we_w_pop_new'])
df_safe['total_weekend_pop'] = df_safe['we_v_pop_new'] + df_safe['we_w_pop_new']
df_safe['v_pop_cv'] = df_safe['v_pop_sd'] / df_safe[['wd_v_pop_new','we_v_pop_new']].mean(axis=1)
df_safe['sd_ratio_v_w'] = df_safe['v_pop_sd'] / df_safe['w_pop_sd']

# 3. Min-Max 정규화
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

for col in ['v_pop_diff','v_pop_ratio','weekend_pure_v_ratio','total_weekend_pop','v_pop_cv','sd_ratio_v_w']:
    df_safe[col+'_norm'] = minmax(df_safe[col])

# 4. 4개 모델 스코어
df_safe['모델1_SD비율단독'] = df_safe['sd_ratio_v_w_norm']
df_safe['모델2_야외특화급증'] = 0.5*df_safe['sd_ratio_v_w_norm'] + 0.5*df_safe['v_pop_diff_norm']
df_safe['모델3_절대밀집'] = 0.4*df_safe['total_weekend_pop_norm'] + 0.3*df_safe['weekend_pure_v_ratio_norm'] + 0.3*df_safe['sd_ratio_v_w_norm']
df_safe['모델4_피크폭증'] = 0.6*df_safe['v_pop_cv_norm'] + 0.4*df_safe['total_weekend_pop_norm']

print("\n모델별 Top3:")
for m in ['모델1_SD비율단독','모델2_야외특화급증','모델3_절대밀집','모델4_피크폭증']:
    print(f"\n[{m}]")
    print(df_safe.nlargest(3, m)[['grid_id', m]])

df_safe.to_csv('shading_priority_4models.csv', index=False, encoding='utf-8-sig')
print("\n저장 완료: shading_priority_4models.csv")

df_safe 격자 수: 10875 / 11153

모델별 Top3:

[모델1_SD비율단독]
      grid_id  모델1_SD비율단독
7680    44162    1.000000
7359    43237    0.933381
5693    38583    0.843945

[모델2_야외특화급증]
      grid_id  모델2_야외특화급증
7680    44162    0.644093
7359    43237    0.608575
5693    38583    0.567009

[모델3_절대밀집]
      grid_id  모델3_절대밀집
477     23238  0.605310
7680    44162  0.537445
478     23240  0.521665

[모델4_피크폭증]
      grid_id  모델4_피크폭증
8053    45195  0.602262
7957    44930  0.600372
3277    31618  0.600176

저장 완료: shading_priority_4models.csv


In [19]:
## 10개 곤란구간 buffer 안에 4개 모델 상위 격자가 포함되는지 확인

grid_04_5186 = grid_04.to_crs(epsg=5186)
df_safe_grid = df_safe.merge(grid_04_5186[['grid_id','geometry']], on='grid_id', how='left')
df_safe_grid = gpd.GeoDataFrame(df_safe_grid, geometry='geometry', crs='EPSG:5186')

# 곤란구간 buffer와 격자 중심점 교차 확인
overlap_check = gpd.sjoin(df_safe_grid, difficult_buffer[['link_id','link_hsi','buffer_geom']].set_geometry('buffer_geom'), how='inner', predicate='intersects')

print(f"곤란구간 buffer 안에 포함된 격자 수: {len(overlap_check)}")
print()

models = ['모델1_SD비율단독','모델2_야외특화급증','모델3_절대밀집','모델4_피크폭증']
for m in models:
    top10pct = df_safe[m].quantile(0.9)
    high_score_in_buffer = overlap_check[overlap_check[m] >= top10pct]
    print(f"[{m}] 상위10%(>={top10pct:.3f}) 기준, 곤란구간 buffer 안에 있는 격자: {len(high_score_in_buffer)}개")
    if len(high_score_in_buffer) > 0:
        print(high_score_in_buffer[['grid_id','link_id','link_hsi',m]].sort_values(m, ascending=False))
    print()

곤란구간 buffer 안에 포함된 격자 수: 32

[모델1_SD비율단독] 상위10%(>=0.001) 기준, 곤란구간 buffer 안에 있는 격자: 0개

[모델2_야외특화급증] 상위10%(>=0.157) 기준, 곤란구간 buffer 안에 있는 격자: 19개
      grid_id    link_id  link_hsi  모델2_야외특화급증
5675    39162  571542735  0.821341    0.189498
5675    39162  571500575  0.865877    0.189498
5803    39522  571500274  0.902470    0.180845
5802    39520  571500274  0.902470    0.174402
5952    39930  571541909  0.904698    0.173753
5758    39375  571541906  0.817914    0.172111
5951    39928  571541909  0.904698    0.168365
5759    39376  571541906  0.817914    0.167505
5891    39756  571500274  0.902470    0.167117
5891    39756  571541909  0.904698    0.167117
5716    39302  571500575  0.865877    0.164549
5849    39677  571500274  0.902470    0.164033
5512    38709  571500307  0.899820    0.162423
5512    38709  571541307  0.887483    0.162423
5892    39757  571500274  0.902470    0.160065
5892    39757  571541909  0.904698    0.160065
5848    39675  571500274  0.902470    0.158867
5551    3

In [21]:
## 적응시설 정보 다시 불러오기

axis_summary_final = pd.read_csv('difficult_segments_3axis_profile_final10.csv')
axis_summary_final['link_id'] = axis_summary_final['link_id'].astype(str)

facility_info = axis_summary_final[['link_id','적응시설_총합']].copy()

print(facility_info)

     link_id   적응시설_총합
0  571500274  0.000000
1  571500307  0.500000
2  571500575  0.000000
3  571541307  0.666667
4  571541880  0.000000
5  571541906  0.000000
6  571541909  0.000000
7  571542735  0.000000
8  571549132  0.000000
9  571551868  0.000000


In [23]:
## 5단계 최종 스코어링

def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

# ① 취약인구노출량: 10개 곤란구간별 모델2 점수 평균 (buffer 안 격자 기준)
vuln_pop_by_link = overlap_check.groupby('link_id')['모델2_야외특화급증'].mean().reset_index()
vuln_pop_by_link.columns = ['link_id', '취약인구노출량_raw']
vuln_pop_by_link['link_id'] = vuln_pop_by_link['link_id'].astype(str)

# ② 기존시설미비도: 그림자(11시+17시 평균) + 기존 적응시설 결합
shade_avg = result[['link_id','그늘비율_11시','그늘비율_17시']].copy()
shade_avg['link_id'] = shade_avg['link_id'].astype(str)
shade_avg['자연그늘_평균'] = shade_avg[['그늘비율_11시','그늘비율_17시']].mean(axis=1)

# 최종 결합
final_score = difficult_10[['link_id','link_hsi']].copy()
final_score['link_id'] = final_score['link_id'].astype(str)
final_score = final_score.merge(vuln_pop_by_link, on='link_id', how='left')
final_score = final_score.merge(shade_avg[['link_id','자연그늘_평균']], on='link_id', how='left')
final_score = final_score.merge(facility_info, on='link_id', how='left')

final_score['취약인구노출량_raw'] = final_score['취약인구노출량_raw'].fillna(0)
final_score['자연그늘_평균'] = final_score['자연그늘_평균'].fillna(0)
final_score['적응시설_총합'] = final_score['적응시설_총합'].fillna(0)

# ① 곤란구간등급 정규화
final_score['곤란구간등급_norm'] = minmax(final_score['link_hsi'])

# ② 취약인구노출량 정규화
final_score['취약인구노출량_norm'] = minmax(final_score['취약인구노출량_raw'])

# ③ 기존시설미비도 = 1 - (인공시설 정규화 + 자연그늘 정규화를 결합한 "차폐충족도")
final_score['적응시설_norm'] = minmax(final_score['적응시설_총합'])
final_score['차폐충족도'] = 0.5*final_score['적응시설_norm'] + 0.5*final_score['자연그늘_평균']
final_score['기존시설미비도_norm'] = 1 - final_score['차폐충족도']

# 최종 우선순위 스코어
final_score['우선순위_스코어'] = (
    final_score['곤란구간등급_norm'] *
    final_score['취약인구노출량_norm'] *
    final_score['기존시설미비도_norm']
)

result_final = final_score[[
    'link_id','link_hsi','곤란구간등급_norm','취약인구노출량_norm',
    '적응시설_총합','자연그늘_평균','기존시설미비도_norm','우선순위_스코어'
]].sort_values('우선순위_스코어', ascending=False).reset_index(drop=True)

print(result_final)

result_final.to_csv('priority_score_final_v2.csv', index=False, encoding='utf-8-sig')
print("\n저장 완료: priority_score_final_v2.csv")

     link_id  link_hsi  곤란구간등급_norm  취약인구노출량_norm   적응시설_총합   자연그늘_평균  \
0  571500274  0.902470     0.872381      0.924827  0.000000  0.658218   
1  571541909  0.904698     0.893210      0.917155  0.000000  0.706078   
2  571500575  0.865877     0.530142      0.825249  0.000000  0.239692   
3  571541307  0.887483     0.732213      0.609751  0.666667  0.055336   
4  571500307  0.899820     0.847592      0.273925  0.500000  0.520294   
5  571541906  0.817914     0.081558      1.000000  0.000000  0.580018   
6  571542735  0.821341     0.113612      0.532625  0.000000  0.280790   
7  571551868  0.822205     0.121698      0.356619  0.000000  0.000000   
8  571541880  0.916116     1.000000      0.000000  0.000000  0.537394   
9  571549132  0.809193     0.000000      0.151507  0.000000  0.052251   

   기존시설미비도_norm  우선순위_스코어  
0      0.670891  0.541276  
1      0.646961  0.529999  
2      0.880154  0.385066  
3      0.472332  0.210881  
4      0.364853  0.084710  
5      0.709991  0.057906  


In [24]:
## 최종 우선순위 지도 시각화

import folium

coords_full = pd.read_csv('difficult_segments_coords_full.csv')
coords_full['link_id'] = coords_full['link_id'].astype(str)

result_final_geo = result_final.merge(coords_full[['link_id','mid_lat','mid_lon']], on='link_id', how='left')
result_final_geo.index = result_final_geo.index + 1  # 1위부터

m_final = folium.Map(location=[37.385, 127.115], zoom_start=13, tiles='cartodbpositron')

def get_color(rank):
    if rank <= 3:
        return 'red'
    elif rank <= 6:
        return 'orange'
    else:
        return 'lightgray'

for rank, row in result_final_geo.iterrows():
    if pd.isna(row['mid_lat']):
        continue
    folium.CircleMarker(
        location=[row['mid_lat'], row['mid_lon']],
        radius=18 - rank,
        color=get_color(rank),
        fill=True,
        fill_opacity=0.75,
        popup=folium.Popup(
            f"<b>{rank}순위</b><br>"
            f"link_id: {row['link_id']}<br>"
            f"HSI: {row['link_hsi']:.3f}<br>"
            f"취약인구노출: {row['취약인구노출량_norm']:.3f}<br>"
            f"기존시설미비도: {row['기존시설미비도_norm']:.3f}<br>"
            f"<b>우선순위 스코어: {row['우선순위_스코어']:.3f}</b>",
            max_width=250
        )
    ).add_to(m_final)
    folium.map.Marker(
        [row['mid_lat'], row['mid_lon']],
        icon=folium.DivIcon(html=f"<div style='font-size:13pt;color:black;font-weight:bold;text-shadow:1px 1px 2px white'>{rank}</div>")
    ).add_to(m_final)

m_final.save('priority_map_final.html')
print("저장 완료: priority_map_final.html")
m_final

저장 완료: priority_map_final.html


In [25]:
## 16개 동선 거리차이 분포 확인

alpha_10_check = alpha_wide_df[alpha_wide_df['alpha']==10.0][['목적지','사분면','거리차이_m']].sort_values('거리차이_m', ascending=False)
print(alpha_10_check)

print()
print("기술통계:")
print(alpha_10_check['거리차이_m'].describe())

NameError: name 'alpha_wide_df' is not defined

In [26]:
## 저장된 alpha_scan_results.csv 불러와서 확인

alpha_wide_df = pd.read_csv('alpha_scan_results.csv')

alpha_10_check = alpha_wide_df[alpha_wide_df['alpha']==10.0][['목적지','사분면','거리차이_m']].sort_values('거리차이_m', ascending=False)
print(alpha_10_check)

print()
print("기술통계:")
print(alpha_10_check['거리차이_m'].describe())

     목적지       사분면  거리차이_m
175  판교역  4사분면(남동)   424.3
172  판교역  1사분면(북동)   301.3
164   서현  1사분면(북동)   118.4
165   서현  2사분면(북서)    76.2
167   서현  4사분면(남동)    39.3
161   미금  2사분면(북서)    32.0
173  판교역  2사분면(북서)    27.5
170   야탑  3사분면(남서)    12.9
171   야탑  4사분면(남동)     3.0
168   야탑  1사분면(북동)     2.0
174  판교역  3사분면(남서)     1.9
166   서현  3사분면(남서)     1.0
160   미금  1사분면(북동)     0.0
162   미금  3사분면(남서)     0.0
163   미금  4사분면(남동)     0.0
169   야탑  2사분면(북서)     0.0

기술통계:
count     16.00000
mean      64.98750
std      122.90157
min        0.00000
25%        0.75000
50%        7.95000
75%       48.52500
max      424.30000
Name: 거리차이_m, dtype: float64


In [27]:
## 조건 A: 최단경로 HSI, 회피경로(α=10) HSI 계산

# routes_result_v2_df(최단경로) + alpha_wide_df(α=10 회피경로) 불러오기
routes_result_v2_df = pd.read_csv('shortest_paths_v2_final.csv')
alpha_wide_df = pd.read_csv('alpha_scan_results.csv')

# α=10 결과만 추출
alpha_10 = alpha_wide_df[alpha_wide_df['alpha']==10.0].copy()

print(routes_result_v2_df.columns.tolist())
print()
print(alpha_10.columns.tolist())

['목적지', '사분면', '출발지_grid_id', '출발지_노드', '목적지_노드', '경로_길이_m', '경로_노드수', '경로', '직선거리_m', '우회율_%']

['alpha', '목적지', '사분면', '최단경로_거리_m', '최단경로_HSI', '회피경로_거리_m', '회피경로_HSI', '거리차이_m', '우회율_%', 'HSI차이', '저감률_%']


In [28]:
## 조건 A 판정: 최단경로 HSI, 회피경로(α=10) HSI 둘 다 상위 20% 이상

alpha_10 = alpha_wide_df[alpha_wide_df['alpha']==10.0].copy()

hsi_threshold_route = alpha_10['최단경로_HSI'].quantile(0.8)
print(f"최단경로 HSI 상위20% 임계값: {hsi_threshold_route:.3f}")

hsi_threshold_avoid = alpha_10['회피경로_HSI'].quantile(0.8)
print(f"회피경로 HSI 상위20% 임계값: {hsi_threshold_avoid:.3f}")

alpha_10['조건A'] = (alpha_10['최단경로_HSI'] >= hsi_threshold_route) & (alpha_10['회피경로_HSI'] >= hsi_threshold_avoid)
alpha_10['조건B'] = alpha_10['거리차이_m'] >= 300

alpha_10['곤란구간_유형'] = alpha_10.apply(
    lambda r: ('A' if r['조건A'] else '') + ('B' if r['조건B'] else ''), axis=1
)
alpha_10['곤란구간_최종'] = alpha_10['조건A'] | alpha_10['조건B']

print()
print(alpha_10[['목적지','사분면','최단경로_HSI','회피경로_HSI','거리차이_m','조건A','조건B','곤란구간_최종']].sort_values('곤란구간_최종', ascending=False))

print()
print("최종 곤란구간(동선) 수:", alpha_10['곤란구간_최종'].sum(), "/", len(alpha_10))

최단경로 HSI 상위20% 임계값: 0.710
회피경로 HSI 상위20% 임계값: 0.684

     목적지       사분면  최단경로_HSI  회피경로_HSI  거리차이_m    조건A    조건B  곤란구간_최종
165   서현  2사분면(북서)     0.738     0.684    76.2   True  False     True
166   서현  3사분면(남서)     0.861     0.846     1.0   True  False     True
168   야탑  1사분면(북동)     0.710     0.708     2.0   True  False     True
172  판교역  1사분면(북동)     0.684     0.519   301.3  False   True     True
175  판교역  4사분면(남동)     0.720     0.519   424.3  False   True     True
160   미금  1사분면(북동)     0.658     0.658     0.0  False  False    False
161   미금  2사분면(북서)     0.703     0.633    32.0  False  False    False
162   미금  3사분면(남서)     0.618     0.618     0.0  False  False    False
163   미금  4사분면(남동)     0.618     0.618     0.0  False  False    False
164   서현  1사분면(북동)     0.650     0.569   118.4  False  False    False
167   서현  4사분면(남동)     0.692     0.629    39.3  False  False    False
169   야탑  2사분면(북서)     0.697     0.697     0.0  False  False    False
170   야탑  3사분면(남서)     0.694     0.57

In [29]:
## 5개 곤란구간 동선 시각화 - 최단경로 vs 회피경로

import folium

difficult_routes = alpha_10[alpha_10['곤란구간_최종']].copy()
print(difficult_routes[['목적지','사분면','조건A','조건B']])

# 최단경로 원본 정보 다시 불러오기 (경로 노드리스트 필요)
routes_v2_raw = pd.read_csv('shortest_paths_v2_final.csv')
routes_v2_raw['경로'] = routes_v2_raw['경로'].apply(lambda x: x.split('|') if isinstance(x, str) else [])

# destinations_v2, all_node_coords_4326 등은 12번 노트북 재실행 필요 - 없으면 아래서 에러날 수 있음
print("\n필요 변수 존재 확인:")
print("destinations_v2:", 'destinations_v2' in dir())
print("all_node_coords_4326:", 'all_node_coords_4326' in dir())

     목적지       사분면    조건A    조건B
165   서현  2사분면(북서)   True  False
166   서현  3사분면(남서)   True  False
168   야탑  1사분면(북동)   True  False
172  판교역  1사분면(북동)  False   True
175  판교역  4사분면(남동)  False   True

필요 변수 존재 확인:
destinations_v2: False
all_node_coords_4326: False


In [30]:
## 필요 변수 재구성

destinations_v2 = {
    '야탑': (127.12873, 37.41131),
    '서현': (127.12397, 37.38474),
    '미금': (127.10891, 37.35016),
    '판교역': (127.10979, 37.39614),
}

# 그래프 재구성 (03번+OSM 인도)
import networkx as nx
from shapely.geometry import Point

road_gdf = gpd.read_file('03._상세도로망_lev6.geojson')
osm_roads = gpd.read_file('gis_osm_roads_free_1.shp', bbox=(127.05, 37.33, 127.16, 37.43))
footway_types = ['footway', 'path', 'pedestrian', 'steps']
sidewalk = osm_roads[osm_roads['fclass'].isin(footway_types)].copy()

G = nx.Graph()
for idx, row in road_gdf.iterrows():
    G.add_edge(row['up_f_node'], row['up_t_node'], weight=row['length']*1000, link_id=row['link_id'], source='03번')

sidewalk_5186 = sidewalk.to_crs(epsg=5186)
for idx, row in sidewalk_5186.iterrows():
    coords = list(row.geometry.coords)
    if len(coords) < 2:
        continue
    G.add_edge(f"osm_{idx}_start", f"osm_{idx}_end", weight=row.geometry.length,
               link_id=f"osm_{row['osm_id']}", fclass=row['fclass'], source='OSM_인도')

# 노드 좌표 (03번+OSM)
road_gdf_5186 = road_gdf.to_crs(epsg=5186)
node03_coords = {}
for idx, row in road_gdf_5186.iterrows():
    coords = list(row.geometry.coords)
    node03_coords[row['up_f_node']] = Point(coords[0])
    node03_coords[row['up_t_node']] = Point(coords[-1])

osm_node_coords = {}
for idx, row in sidewalk_5186.iterrows():
    coords = list(row.geometry.coords)
    if len(coords) < 2:
        continue
    osm_node_coords[f"osm_{idx}_start"] = Point(coords[0])
    osm_node_coords[f"osm_{idx}_end"] = Point(coords[-1])

# 스냅 연결
from scipy.spatial import cKDTree
node03_ids_arr = list(node03_coords.keys())
node03_pts_arr = np.array([[p.x, p.y] for p in node03_coords.values()])
tree = cKDTree(node03_pts_arr)
osm_ids_arr = list(osm_node_coords.keys())
osm_pts_arr = np.array([[p.x, p.y] for p in osm_node_coords.values()])
dists, idxs = tree.query(osm_pts_arr)
for i, (osm_id, dist, idx) in enumerate(zip(osm_ids_arr, dists, idxs)):
    if dist <= 100.0:
        G.add_edge(osm_id, node03_ids_arr[idx], weight=dist, source='snap_connection')

largest_cc = max(nx.connected_components(G), key=len)
G_main = G.subgraph(largest_cc).copy()

# 전체 노드 좌표 (4326 변환)
all_node_coords = {**node03_coords, **osm_node_coords}
all_node_coords_4326 = {}
for n, pt in all_node_coords.items():
    if n in G_main.nodes:
        gs = gpd.GeoSeries([pt], crs='EPSG:5186').to_crs(epsg=4326)
        all_node_coords_4326[n] = gs.iloc[0]

print(f"그래프 노드: {G_main.number_of_nodes()}, 엣지: {G_main.number_of_edges()}")
print(f"좌표 확보: {len(all_node_coords_4326)}")

그래프 노드: 7243, 엣지: 9930
좌표 확보: 7243


In [31]:
## α=10 회피경로 재계산 (곤란구간 5개 대상)

# link_hsi_summary 다시 불러오기
link_hsi_summary = pd.read_csv('link_hsi_summary.csv')
mean_hsi = link_hsi_summary['link_hsi'].mean()
link_hsi_map = dict(zip(link_hsi_summary['link_id'].astype(str), link_hsi_summary['link_hsi']))

for u, v, data in G_main.edges(data=True):
    lid = str(data.get('link_id', ''))
    source = data.get('source', '')
    if source == 'snap_connection':
        neighbor_hsi = None
        for node in [u, v]:
            for neighbor in G_main.neighbors(node):
                if neighbor in (u, v):
                    continue
                edge = G_main.get_edge_data(node, neighbor)
                if edge and edge.get('source') != 'snap_connection' and str(edge.get('link_id','')) in link_hsi_map:
                    neighbor_hsi = link_hsi_map[str(edge['link_id'])]
                    break
            if neighbor_hsi is not None:
                break
        G_main[u][v]['link_hsi'] = neighbor_hsi if neighbor_hsi is not None else mean_hsi
    else:
        G_main[u][v]['link_hsi'] = link_hsi_map.get(lid, mean_hsi)

def build_cost_graph(G, alpha):
    G_cost = G.copy()
    for u, v, data in G_cost.edges(data=True):
        data['cost'] = data['weight'] * (1 + alpha * data.get('link_hsi', mean_hsi))
    return G_cost

G_cost10 = build_cost_graph(G_main, 10.0)

# 목적지 노드 매칭 필요 (grid_master 필요)
grid_master = gpd.read_file('grid_master_final.shp')
grid_master_5186 = grid_master.to_crs(epsg=5186)

dest_grid_ids = {'야탑': 23412, '서현': 38207, '미금': 50277, '판교역': 30721}

def find_nearest_node(grid_id, grid_gdf_5186, node_coords_5186):
    row = grid_gdf_5186[grid_gdf_5186['grid_id'] == grid_id].iloc[0]
    centroid = row.geometry.centroid
    node_ids = list(node_coords_5186.keys())
    node_pts = np.array([[p.x, p.y] for p in node_coords_5186.values()])
    tree2 = cKDTree(node_pts)
    dist, idx = tree2.query([centroid.x, centroid.y])
    return node_ids[idx], dist

all_node_coords_5186 = {**node03_coords, **osm_node_coords}
all_node_coords_5186 = {k:v for k,v in all_node_coords_5186.items() if k in G_main.nodes}

dest_nodes = {}
for name, gid in dest_grid_ids.items():
    node, dist = find_nearest_node(gid, grid_master_5186, all_node_coords_5186)
    dest_nodes[name] = node

print("목적지 노드 매칭 완료:", dest_nodes)

목적지 노드 매칭 완료: {'야탑': 353073, '서현': 353941, '미금': 'osm_11517_start', '판교역': 337723}


In [32]:
## 5개 곤란구간 동선 - 출발지 매칭 + 경로계산 + 시각화

# routes_quadrant_top 재구성 필요 (출발지 grid_id 확보용)
routes_v2_raw = pd.read_csv('shortest_paths_v2_final.csv')

difficult_routes_info = alpha_10[alpha_10['곤란구간_최종']][['목적지','사분면']].merge(
    routes_v2_raw[['목적지','사분면','출발지_grid_id']], on=['목적지','사분면'], how='left'
)
print(difficult_routes_info)

# 출발지 노드 매칭
origin_nodes = {}
for _, row in difficult_routes_info.iterrows():
    gid = row['출발지_grid_id']
    if gid not in origin_nodes:
        node, dist = find_nearest_node(gid, grid_master_5186, all_node_coords_5186)
        origin_nodes[gid] = node

print("\n출발지 노드 매칭 완료:", origin_nodes)

   목적지       사분면  출발지_grid_id
0   서현  2사분면(북서)        33034
1   서현  3사분면(남서)        40242
2   야탑  1사분면(북동)        23441
3  판교역  1사분면(북동)        26512
4  판교역  4사분면(남동)        33449

출발지 노드 매칭 완료: {33034: 'osm_10123_start', 40242: 'osm_9258_start', 23441: 'osm_944_end', 26512: 336462, 33449: 'osm_9734_end'}


In [33]:
## 최단경로 vs 회피경로 계산 + 지도 시각화

import folium

routes_compare = []
for _, row in difficult_routes_info.iterrows():
    dest_name = row['목적지']
    origin_gid = row['출발지_grid_id']
    origin_node = origin_nodes[origin_gid]
    dest_node = dest_nodes[dest_name]

    shortest_path = nx.shortest_path(G_main, source=origin_node, target=dest_node, weight='weight')
    avoid_path = nx.shortest_path(G_cost10, source=origin_node, target=dest_node, weight='cost')

    routes_compare.append({
        '목적지': dest_name, '사분면': row['사분면'],
        '최단경로': shortest_path, '회피경로': avoid_path
    })

colors = {'야탑':'red', '서현':'blue', '미금':'green', '판교역':'purple'}

m_diff = folium.Map(location=[37.39, 127.115], zoom_start=13, tiles='cartodbpositron')

for name, (lon, lat) in destinations_v2.items():
    folium.Marker(location=[lat, lon], popup=f"{name}", icon=folium.Icon(color=colors[name], icon='star')).add_to(m_diff)

for r in routes_compare:
    dest_name = r['목적지']
    color = colors[dest_name]

    short_coords = [[all_node_coords_4326[n].y, all_node_coords_4326[n].x] for n in r['최단경로'] if n in all_node_coords_4326]
    avoid_coords = [[all_node_coords_4326[n].y, all_node_coords_4326[n].x] for n in r['회피경로'] if n in all_node_coords_4326]

    folium.PolyLine(short_coords, color=color, weight=4, opacity=0.9,
                     popup=f"{dest_name}({r['사분면']}) 최단경로").add_to(m_diff)
    folium.PolyLine(avoid_coords, color=color, weight=3, opacity=0.6, dash_array='10,6',
                     popup=f"{dest_name}({r['사분면']}) 회피경로").add_to(m_diff)

m_diff.save('difficult_routes_comparison.html')
print("저장 완료: difficult_routes_comparison.html")
m_diff

저장 완료: difficult_routes_comparison.html


In [34]:
## 16개 동선 전체 - 최단경로 vs 회피경로(α=10) 지도

routes_compare_all = []
for _, row in routes_v2_raw.iterrows():
    dest_name = row['목적지']
    origin_gid = row['출발지_grid_id']

    if origin_gid not in origin_nodes:
        node, dist = find_nearest_node(origin_gid, grid_master_5186, all_node_coords_5186)
        origin_nodes[origin_gid] = node

    origin_node = origin_nodes[origin_gid]
    dest_node = dest_nodes[dest_name]

    try:
        shortest_path = nx.shortest_path(G_main, source=origin_node, target=dest_node, weight='weight')
        avoid_path = nx.shortest_path(G_cost10, source=origin_node, target=dest_node, weight='cost')
        routes_compare_all.append({
            '목적지': dest_name, '사분면': row['사분면'],
            '최단경로': shortest_path, '회피경로': avoid_path
        })
    except nx.NetworkXNoPath:
        pass

m_all = folium.Map(location=[37.385, 127.115], zoom_start=13, tiles='cartodbpositron')

for name, (lon, lat) in destinations_v2.items():
    folium.Marker(location=[lat, lon], popup=f"{name}", icon=folium.Icon(color=colors[name], icon='star')).add_to(m_all)

for r in routes_compare_all:
    color = colors[r['목적지']]
    short_coords = [[all_node_coords_4326[n].y, all_node_coords_4326[n].x] for n in r['최단경로'] if n in all_node_coords_4326]
    avoid_coords = [[all_node_coords_4326[n].y, all_node_coords_4326[n].x] for n in r['회피경로'] if n in all_node_coords_4326]

    folium.PolyLine(short_coords, color=color, weight=3, opacity=0.7,
                     popup=f"{r['목적지']}({r['사분면']}) 최단경로").add_to(m_all)
    folium.PolyLine(avoid_coords, color=color, weight=2, opacity=0.5, dash_array='8,4',
                     popup=f"{r['목적지']}({r['사분면']}) 회피경로(α=10)").add_to(m_all)

m_all.save('all_routes_comparison_alpha10.html')
print("저장 완료: all_routes_comparison_alpha10.html")
m_all

저장 완료: all_routes_comparison_alpha10.html


In [35]:
## 실제로 계산 성공한 동선 개수 확인

print(f"성공한 동선 개수: {len(routes_compare_all)} / 16")
print()
for r in routes_compare_all:
    print(r['목적지'], r['사분면'])

성공한 동선 개수: 16 / 16

미금 1사분면(북동)
미금 2사분면(북서)
미금 3사분면(남서)
미금 4사분면(남동)
서현 1사분면(북동)
서현 2사분면(북서)
서현 3사분면(남서)
서현 4사분면(남동)
야탑 1사분면(북동)
야탑 2사분면(북서)
야탑 3사분면(남서)
야탑 4사분면(남동)
판교역 1사분면(북동)
판교역 2사분면(북서)
판교역 3사분면(남서)
판교역 4사분면(남동)


In [36]:
## 5개 곤란구간 동선의 3축 프로파일 재계산

# 최단경로가 지나가는 모든 링크 추출 (동선 단위 -> 링크 리스트)
def get_link_ids_from_path(path_nodes, G):
    link_ids = []
    for i in range(len(path_nodes) - 1):
        edge = G.get_edge_data(path_nodes[i], path_nodes[i+1])
        if edge and 'link_id' in edge:
            link_ids.append(str(edge['link_id']))
    return link_ids

# 5개 곤란구간 동선의 링크 목록(최단경로 기준, 조건B는 회피경로도 포함)
route_links = {}
for r in routes_compare:
    key = f"{r['목적지']}_{r['사분면']}"
    links_short = get_link_ids_from_path(r['최단경로'], G_main)
    links_avoid = get_link_ids_from_path(r['회피경로'], G_main)
    route_links[key] = list(set(links_short + links_avoid))  # 최단+회피 경로 모두 포함

for k, v in route_links.items():
    print(k, ':', len(v), '개 링크')

서현_2사분면(북서) : 42 개 링크
서현_3사분면(남서) : 14 개 링크
야탑_1사분면(북동) : 14 개 링크
판교역_1사분면(북동) : 38 개 링크
판교역_4사분면(남동) : 30 개 링크


In [37]:
## 16개 동선별 α=10 기준 우회율/저감률/효율지수 표

alpha_10_individual = alpha_wide_df[alpha_wide_df['alpha']==10.0][
    ['목적지','사분면','최단경로_HSI','회피경로_HSI','거리차이_m','우회율_%','HSI차이','저감률_%']
].copy()

alpha_10_individual['효율지수'] = alpha_10_individual['저감률_%'] / alpha_10_individual['우회율_%'].replace(0, np.nan)

print(alpha_10_individual.sort_values('저감률_%', ascending=False))

     목적지       사분면  최단경로_HSI  회피경로_HSI  거리차이_m      우회율_%  HSI차이      저감률_%  \
175  판교역  4사분면(남동)     0.720     0.519   424.3  29.223776  0.201  27.916667   
172  판교역  1사분면(북동)     0.684     0.519   301.3  16.420513  0.165  24.122807   
170   야탑  3사분면(남서)     0.694     0.579    12.9   1.150245  0.115  16.570605   
164   서현  1사분면(북동)     0.650     0.569   118.4   7.857712  0.081  12.461538   
161   미금  2사분면(북서)     0.703     0.633    32.0   2.576282  0.070   9.957326   
167   서현  4사분면(남동)     0.692     0.629    39.3   2.200448  0.063   9.104046   
165   서현  2사분면(북서)     0.738     0.684    76.2   3.931483  0.054   7.317073   
173  판교역  2사분면(북서)     0.709     0.670    27.5   1.397358  0.039   5.500705   
171   야탑  4사분면(남동)     0.707     0.680     3.0   0.269542  0.027   3.818953   
174  판교역  3사분면(남서)     0.681     0.661     1.9   0.145549  0.020   2.936858   
166   서현  3사분면(남서)     0.861     0.846     1.0   0.072480  0.015   1.742160   
168   야탑  1사분면(북동)     0.710     0.708     2.0   0.1

In [38]:
## 5개 동선의 3축 프로파일 계산

# grid_HSI_selected_indicators_v2.csv 불러오기 (원본 지표값)
hsi_full = pd.read_csv('grid_HSI_selected_indicators_v2.csv')
hsi_full['adaptive_facility_total'] = (
    hsi_full['park_cnt'] + hsi_full['shade_cnt'] +
    hsi_full['cooling_shelter_cnt'] + hsi_full['smart_shelter_cnt']
)

# 03번+OSM 통합 geometry (all_links_gdf) - 앞서 만든 거 재사용
# route_links 딕셔너리(동선별 link_id 리스트)는 이미 있음

route_profiles = []
for key, link_ids in route_links.items():
    # 해당 동선의 링크 geometry 모으기
    links_geom = all_links_gdf[all_links_gdf['link_id'].astype(str).isin(link_ids)].copy()
    if len(links_geom) == 0:
        continue

    links_geom_5186 = links_geom.to_crs(epsg=5186)
    links_geom_5186['geometry'] = links_geom_5186.geometry.buffer(12)

    grid_5186_route = gpd.GeoDataFrame(
        hsi_full,
        geometry=gpd.GeoSeries.from_wkt(hsi_full['geometry']) if hsi_full['geometry'].dtype == object else hsi_full['geometry'],
        crs='EPSG:4326'
    ).to_crs(epsg=5186)

    overlay_route = gpd.overlay(links_geom_5186[['link_id','geometry']], grid_5186_route, how='intersection')

    profile = {
        '동선': key,
        '링크수': len(link_ids),
        '시가화비율': overlay_route['LC1_100_시가화건조지역_ratio'].mean(),
        '녹지비율': overlay_route['GREEN_ratio_LC'].mean(),
        'NDVI': overlay_route['NDVI_mean_5scene'].mean(),
        '적응시설_평균': overlay_route['adaptive_facility_total'].mean(),
    }
    route_profiles.append(profile)

route_profile_df = pd.DataFrame(route_profiles)
print(route_profile_df)

route_profile_df.to_csv('difficult_routes_3axis_profile.csv', index=False, encoding='utf-8-sig')
print("\n저장 완료: difficult_routes_3axis_profile.csv")

             동선  링크수     시가화비율      녹지비율      NDVI   적응시설_평균
0   서현_2사분면(북서)   42  0.831745  0.161525  0.366142  0.115108
1   서현_3사분면(남서)   14  0.944402  0.054175  0.374973  0.084746
2   야탑_1사분면(북동)   14  0.935686  0.058312  0.344807  0.070175
3  판교역_1사분면(북동)   38  0.767122  0.215456  0.356407  0.107438
4  판교역_4사분면(남동)   30  0.687948  0.269347  0.376158  0.076923

저장 완료: difficult_routes_3axis_profile.csv


In [39]:
## 5개 동선 buffer + 취약인구노출량(팀원 모델2) 재집계

# 5개 동선의 링크 geometry -> buffer
route_buffers = []
for key, link_ids in route_links.items():
    links_geom = all_links_gdf[all_links_gdf['link_id'].astype(str).isin(link_ids)].copy()
    if len(links_geom) == 0:
        continue
    links_geom_5186 = links_geom.to_crs(epsg=5186)
    links_geom_5186['geometry'] = links_geom_5186.geometry.buffer(12)
    dissolved = links_geom_5186.geometry.unary_union
    route_buffers.append({'동선': key, 'geometry': dissolved})

route_buffer_gdf = gpd.GeoDataFrame(route_buffers, geometry='geometry', crs='EPSG:5186')
route_buffer_gdf['buffer_area'] = route_buffer_gdf.geometry.area

print(route_buffer_gdf[['동선','buffer_area']])

             동선   buffer_area
0   서현_2사분면(북서)  87203.123950
1   서현_3사분면(남서)  37500.493967
2   야탑_1사분면(북동)  35188.602710
3  판교역_1사분면(북동)  76027.094411
4  판교역_4사분면(남동)  75394.749669


In [44]:
## 5개 동선의 전체 링크 통합 + 링크 단위 3단 스코어링

all_route_link_ids = list(set([lid for links in route_links.values() for lid in links]))
print(f"5개 동선 통합 링크 수(중복제거): {len(all_route_link_ids)}")

# 링크별 3축 프로파일 (기존 get_link_level_profile 함수 재사용, 전체 링크 대상)
all_link_profile = get_link_level_profile(all_route_link_ids, all_links_gdf, hsi_full, link_hsi_summary)

# 취약인구노출량 (링크별 buffer)
links_geom_all = all_links_gdf[all_links_gdf['link_id'].astype(str).isin(all_route_link_ids)].copy()
links_geom_all_5186 = links_geom_all.to_crs(epsg=5186)
links_geom_all_5186['geometry'] = links_geom_all_5186.geometry.buffer(12)

vuln_pop_link = gpd.sjoin(df_safe_grid, links_geom_all_5186[['link_id','geometry']], how='inner', predicate='intersects')
vuln_pop_by_link2 = vuln_pop_link.groupby('link_id')['모델2_야외특화급증'].mean().reset_index()
vuln_pop_by_link2.columns = ['link_id', '취약인구노출량_raw']
vuln_pop_by_link2['link_id'] = vuln_pop_by_link2['link_id'].astype(str)

# 그림자 커버리지 (링크별)
def calc_shadow_coverage_link(buffer_gdf, shadow_gdf, label):
    overlay = gpd.overlay(buffer_gdf[['link_id','geometry']], shadow_gdf[['geometry']], how='intersection')
    overlay['shadow_area'] = overlay.geometry.area
    coverage = overlay.groupby('link_id')['shadow_area'].sum().reset_index()
    coverage.columns = ['link_id', f'shadow_area_{label}']
    return coverage

links_geom_all_5186['buffer_area'] = links_geom_all_5186.geometry.area
cov_11_link = calc_shadow_coverage_link(links_geom_all_5186, shadow_11_single, '11시')
cov_17_link = calc_shadow_coverage_link(links_geom_all_5186, shadow_17_single, '17시')

shadow_link_result = links_geom_all_5186[['link_id','buffer_area']].merge(cov_11_link, on='link_id', how='left').merge(cov_17_link, on='link_id', how='left')
shadow_link_result['link_id'] = shadow_link_result['link_id'].astype(str)
shadow_link_result['shadow_area_11시'] = shadow_link_result['shadow_area_11시'].fillna(0)
shadow_link_result['shadow_area_17시'] = shadow_link_result['shadow_area_17시'].fillna(0)
shadow_link_result['그늘비율_11시'] = (shadow_link_result['shadow_area_11시'] / shadow_link_result['buffer_area']).clip(upper=1.0)
shadow_link_result['그늘비율_17시'] = (shadow_link_result['shadow_area_17시'] / shadow_link_result['buffer_area']).clip(upper=1.0)
shadow_link_result['자연그늘_평균'] = shadow_link_result[['그늘비율_11시','그늘비율_17시']].mean(axis=1)

print("완료")

5개 동선 통합 링크 수(중복제거): 120
완료


In [45]:
## 링크 단위 최종 3단 스코어링

def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

final_link_score = all_link_profile[['link_id','link_hsi','시가화비율','녹지비율','적응시설']].copy()
final_link_score['link_id'] = final_link_score['link_id'].astype(str)

final_link_score = final_link_score.merge(vuln_pop_by_link2, on='link_id', how='left')
final_link_score = final_link_score.merge(shadow_link_result[['link_id','자연그늘_평균']], on='link_id', how='left')

final_link_score['취약인구노출량_raw'] = final_link_score['취약인구노출량_raw'].fillna(0)
final_link_score['자연그늘_평균'] = final_link_score['자연그늘_평균'].fillna(0)

final_link_score['곤란구간등급_norm'] = minmax(final_link_score['link_hsi'])
final_link_score['취약인구노출량_norm'] = minmax(final_link_score['취약인구노출량_raw'])
final_link_score['적응시설_norm'] = minmax(final_link_score['적응시설'])
final_link_score['차폐충족도'] = 0.5*final_link_score['적응시설_norm'] + 0.5*final_link_score['자연그늘_평균']
final_link_score['기존시설미비도_norm'] = 1 - final_link_score['차폐충족도']

final_link_score['우선순위_스코어'] = (
    final_link_score['곤란구간등급_norm'] *
    final_link_score['취약인구노출량_norm'] *
    final_link_score['기존시설미비도_norm']
)

result_link_final = final_link_score[[
    'link_id','link_hsi','곤란구간등급_norm','취약인구노출량_norm',
    '적응시설','자연그늘_평균','기존시설미비도_norm','우선순위_스코어'
]].sort_values('우선순위_스코어', ascending=False).reset_index(drop=True)

print(result_link_final.head(15))

result_link_final.to_csv('priority_score_link_final.csv', index=False, encoding='utf-8-sig')
print(f"\n저장 완료: priority_score_link_final.csv (총 {len(result_link_final)}개 링크)")

           link_id  link_hsi  곤란구간등급_norm  취약인구노출량_norm      적응시설   자연그늘_평균  \
0        571556756  0.933145     0.945861      0.994611  0.000000  0.000000   
1        571551880  0.903146     0.901663      0.932060  0.000000  0.090418   
2        571557394  0.869715     0.852407      0.914423  0.000000  0.093165   
3        571556754  0.926950     0.936734      0.782045  0.000000  0.000000   
4        571557831  0.844635     0.815455      0.717588  0.000000  0.000000   
5        571557824  0.844635     0.815455      0.717588  0.000000  0.024554   
6        571541824  0.868961     0.851295      0.803052  0.000000  0.340293   
7   osm_1284737089  0.775436     0.713500      0.776386  0.000000  0.125163   
8        571541825  0.907708     0.908384      0.765796  0.000000  0.552518   
9        572541017  0.841210     0.810408      1.000000  0.166667  0.603172   
10       571500575  0.865877     0.846753      0.663297  0.000000  0.239692   
11       572544942  0.865114     0.845629      0.932

In [46]:
## 상위 링크 좌표 추출

top_links_geom = all_links_gdf[all_links_gdf['link_id'].astype(str).isin(result_link_final.head(15)['link_id'])].copy()
top_links_geom['mid_pt'] = top_links_geom.geometry.interpolate(0.5, normalized=True)
top_links_geom['lat'] = top_links_geom['mid_pt'].y
top_links_geom['lon'] = top_links_geom['mid_pt'].x

top_links_geom['link_id'] = top_links_geom['link_id'].astype(str)
top15_with_coords = result_link_final.head(15).merge(top_links_geom[['link_id','lat','lon']], on='link_id', how='left')

print(top15_with_coords[['link_id','우선순위_스코어','lat','lon']])

top15_with_coords.to_csv('priority_top15_link_coords.csv', index=False, encoding='utf-8-sig')
print("\n저장 완료: priority_top15_link_coords.csv")

           link_id  우선순위_스코어        lat         lon
0        571556756  0.940764  37.396274  127.109930
1        571551880  0.802410  37.396435  127.109699
2        571557394  0.743151  37.398370  127.112766
3        571556754  0.732568  37.396272  127.110244
4        571557831  0.585161  37.396270  127.110361
5        571557824  0.577977  37.396204  127.110450
6        571541824  0.567316  37.387882  127.121105
7   osm_1284737089  0.519284  37.398468  127.112974
8        571541825  0.503461  37.387447  127.121615
9        572541017  0.498466  37.411365  127.129155
10       571500575  0.494337  37.381547  127.116249
11       572544942  0.492215  37.410817  127.128759
12       571501532  0.477829  37.390216  127.122296
13       572501277  0.477202  37.410082  127.128753
14       571557828  0.469734  37.395942  127.110892

저장 완료: priority_top15_link_coords.csv


/tmp/ipykernel_62/718383395.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'interpolate' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  top_links_geom['mid_pt'] = top_links_geom.geometry.interpolate(0.5, normalized=True)


In [47]:
## 5개 곤란구간 동선(120개 링크) buffer로 4개 모델 재검증

# 120개 링크 통합 buffer (아까 만든 links_geom_all_5186 재사용)
overlap_check_v2 = gpd.sjoin(df_safe_grid, links_geom_all_5186[['link_id','geometry']], how='inner', predicate='intersects')

models = ['모델1_SD비율단독','모델2_야외특화급증','모델3_절대밀집','모델4_피크폭증']

print("=== 5개 곤란구간 동선(120개 링크) 기준 재검증 ===\n")
for m in models:
    top10pct = df_safe[m].quantile(0.9)
    high_score_in_buffer = overlap_check_v2[overlap_check_v2[m] >= top10pct]
    n_unique_links = high_score_in_buffer['link_id'].nunique()
    print(f"[{m}] 상위10%(>={top10pct:.3f}) 기준, 곤란구간(120개 링크) 중 겹치는 것: {n_unique_links}개")

=== 5개 곤란구간 동선(120개 링크) 기준 재검증 ===

[모델1_SD비율단독] 상위10%(>=0.001) 기준, 곤란구간(120개 링크) 중 겹치는 것: 29개
[모델2_야외특화급증] 상위10%(>=0.157) 기준, 곤란구간(120개 링크) 중 겹치는 것: 48개
[모델3_절대밀집] 상위10%(>=0.261) 기준, 곤란구간(120개 링크) 중 겹치는 것: 35개
[모델4_피크폭증] 상위10%(>=0.457) 기준, 곤란구간(120개 링크) 중 겹치는 것: 14개
